# 04 — Golden RAGAS Dataset (Production · `gpt-4o` constructor)

> **Phase 3 production build.** Constructor locked to `gpt-4o` after the A/B comparison ([04_notebook_output.md](../docs/output_notes/04_notebook_output.md)).
> New 3-pass prompts adopted ([src/generation/golden_prompts.py](../src/generation/golden_prompts.py)) — structured `selected_chunks`, verbatim `best_gold_context`, tightened `requires_multihop` definition.

## Two-stage run pattern

This notebook supports two execution modes, controlled by the `STAGE` flag in §1:

| `STAGE` | N | Purpose | Cost |
|---|---|---|---|
| `"pilot"` | 50 | Validate the new prompts on a strict subset of the 300-row sample | ~$1.10 |
| `"production"` | 300 | Full build. The 50 pilot rows are cached → only +$5.50 net new | ~$5.50 added |

**Run once with `STAGE = "pilot"`**, inspect quality gates in §11. If gates pass, change to `STAGE = "production"` and re-run — the cache covers the first 50 questions, only the new 250 hit the API.

## Pipeline (staged JSONL — each pass writes its own file)

```
Stage A — stratified sample (300 with seed=42; pilot slices first 50)
    ↓
Stage B — Hybrid RRF retrieval               → data/processed/golden/golden_candidates.jsonl
    ↓
Stage C — Pass 1 evidence selection (T=0)    → data/processed/golden/golden_evidence_selected.jsonl
    ↓
Stage D — Pass 2 reference answer (T=0.2)    → data/processed/golden/golden_with_references.jsonl
    ↓
Stage E — Pass 3 validation (T=0)            → data/processed/golden/golden_validated.jsonl
    ↓
Stage F — Audit + accept/needs_review/drop   → golden_main_accepted.jsonl
                                              + golden_main_needs_review.jsonl
                                              + golden_main_dropped.jsonl
```

**Why staged JSONL?** Each pass reads the previous file and writes the next. If we tighten Pass 3's prompt later, we re-run only Stage E — Stages B/C/D stay cached on disk. Easier to debug, easier to selectively re-run, easier to inspect.

## Quality gates ([docs/todo.md §4](../docs/todo.md))

- Accept rate ≥ 80 %
- JSON malformation < 5 %
- Pass-1 sufficiency ≥ 90 %
- All cited `chunk_id`s resolve in `chunks.parquet`
- `requires_multihop = "yes"` rate < 60 % (Pass 2 prompt has tightened definition)

## Prerequisite

`OPENAI_API_KEY` populated in `.env` (≥$5 credit on the OpenAI account).

## 1. Setup

In [1]:
import sys, os, json, time, textwrap, logging
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.WARNING)

cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

from src.data.embedder import load_bge, best_device
from src.data.indices import load_chroma, load_bm25
from src.retrieval.hybrid import hybrid_top_k_with_text
from src.generation.openai_client import openai_complete_full
from src.generation.golden_prompts import (
    pass1_prompt, pass2_prompt, pass3_prompt,
    parse_json_with_reasoning_leak,
    validate_pass1, validate_pass2, validate_pass3,
)

# === STAGE FLAG: change this to "production" once pilot passes gates ===
STAGE = "production"            # "pilot" → N=50, "production" → N=300
assert STAGE in {"pilot", "production"}

CONSTRUCTOR_MODEL = "gpt-4o"
SEED              = 42
RETRIEVAL_K       = 10
PILOT_N           = 50
PRODUCTION_N      = 300
N = PILOT_N if STAGE == "pilot" else PRODUCTION_N

# Per-pass settings
PASS1_TEMP, PASS1_MAX_TOKENS = 0.0, 4096
PASS2_TEMP, PASS2_MAX_TOKENS = 0.2, 4096
PASS3_TEMP, PASS3_MAX_TOKENS = 0.0, 2048

CHUNKS_PATH       = REPO_ROOT / "data" / "processed" / "chunks.parquet"
MEDQA_4OPT_PATH   = REPO_ROOT / "data" / "processed" / "medqa_4opt.parquet"
CHROMA_DIR        = REPO_ROOT / "data" / "indices" / "chroma_textbooks"
BM25_PATH         = REPO_ROOT / "data" / "indices" / "bm25.pkl"
CACHE_DIR         = REPO_ROOT / "data" / "cache"
GOLDEN_DIR        = REPO_ROOT / "data" / "processed" / "golden"
GOLDEN_DIR.mkdir(parents=True, exist_ok=True)

# Staged-pipeline JSONL files (in data/processed/golden/)
CANDIDATES_PATH   = GOLDEN_DIR / "golden_candidates.jsonl"
EVIDENCE_PATH     = GOLDEN_DIR / "golden_evidence_selected.jsonl"
REFERENCES_PATH   = GOLDEN_DIR / "golden_with_references.jsonl"
VALIDATED_PATH    = GOLDEN_DIR / "golden_validated.jsonl"
ACCEPTED_PATH     = GOLDEN_DIR / "golden_main_accepted.jsonl"
NEEDS_REVIEW_PATH = GOLDEN_DIR / "golden_main_needs_review.jsonl"
DROPPED_PATH      = GOLDEN_DIR / "golden_main_dropped.jsonl"

print(f"Stage:        {STAGE}  (N={N})")
print(f"Constructor:  {CONSTRUCTOR_MODEL}")
print(f"Cache dir:    {CACHE_DIR}")
print(f"Golden dir:   {GOLDEN_DIR}")


Stage:        production  (N=300)
Constructor:  gpt-4o
Cache dir:    /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/cache
Golden dir:   /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/golden


## 2. Verify `OPENAI_API_KEY`

In [2]:
openai_key = os.environ.get("OPENAI_API_KEY")
assert openai_key, (
    "OPENAI_API_KEY missing — populate .env at the repo root and rerun. "
    "Get a key at https://platform.openai.com/api-keys (~$5 minimum credit)."
)
print(f"OPENAI_API_KEY: present (length {len(openai_key)})")

from openai import OpenAI
all_models = {m.id for m in OpenAI(api_key=openai_key).models.list().data}
assert CONSTRUCTOR_MODEL in all_models, f"{CONSTRUCTOR_MODEL!r} not reachable"
print(f"Constructor model {CONSTRUCTOR_MODEL!r}: reachable ({len(all_models)} models visible)")


OPENAI_API_KEY: present (length 164)
Constructor model 'gpt-4o': reachable (132 models visible)


## 3. Stage A — Stratified sample

Sample the **full 300** with seed=42 first, then slice the first `N` rows. This makes the pilot a strict subset of the production sample, so the cached pilot outputs are reused verbatim when scaling up.

In [4]:
medqa = pd.read_parquet(MEDQA_4OPT_PATH)
medqa["q_words"] = medqa["question"].str.split().str.len()

def length_bucket(n: int) -> str:
    if n <= 120: return "short"
    if n <= 200: return "medium"
    return "long"

medqa["length_bucket"] = medqa["q_words"].apply(length_bucket)

# Stratified sample of 300 — force 60 long-vignettes (~5 / book of 18 → 30 each meta_info)
N_LONG_PER_META = 30
N_REMAINDER = PRODUCTION_N - 2 * N_LONG_PER_META  # 240

rng = np.random.RandomState(SEED)
parts = []
for meta in ["step1", "step2&3"]:
    pool = medqa[(medqa["meta_info"] == meta) & (medqa["length_bucket"] == "long")]
    parts.append(pool.sample(N_LONG_PER_META, random_state=rng))

remainder_pool = medqa[medqa["length_bucket"].isin(["short", "medium"])]
strata = remainder_pool.groupby(["meta_info", "length_bucket"])
total_remainder_pop = len(remainder_pool)
allocations = {}
for (meta, bucket), grp in strata:
    allocations[(meta, bucket)] = max(1, round(len(grp) / total_remainder_pop * N_REMAINDER))
diff = N_REMAINDER - sum(allocations.values())
if diff != 0:
    biggest_key = max(allocations, key=lambda k: allocations[k])
    allocations[biggest_key] += diff
for (meta, bucket), nq in allocations.items():
    pool = remainder_pool[(remainder_pool["meta_info"] == meta) &
                          (remainder_pool["length_bucket"] == bucket)]
    parts.append(pool.sample(nq, random_state=rng))

# Shuffle once with the same seed so pilot and production share the same first 50
full_sample = pd.concat(parts).sample(frac=1, random_state=rng).reset_index(drop=True)
assert len(full_sample) == PRODUCTION_N, f"expected {PRODUCTION_N} got {len(full_sample)}"

# Slice to current stage
dev_sample = full_sample.head(N).reset_index(drop=True)
print(f"Full stratified sample: {len(full_sample)} rows ({STAGE} slice = first {N})")
print("\nStrata breakdown for current stage:")
print(dev_sample.groupby(["meta_info", "length_bucket"]).size().unstack(fill_value=0))
print(f"\nLong-vignette count: {(dev_sample.length_bucket == 'long').sum()} / {N}")


Full stratified sample: 300 rows (production slice = first 300)

Strata breakdown for current stage:
length_bucket  long  medium  short
meta_info                         
step1            30      31    105
step2&3          30      65     39

Long-vignette count: 60 / 300


## 4. Load shared infrastructure

In [5]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
chunk_text_by_id = dict(zip(chunks_df.chunk_id, chunks_df.text))
chunk_book_by_id = dict(zip(chunks_df.chunk_id, chunks_df.book_name))

chroma_coll = load_chroma(CHROMA_DIR)
bm25_payload = load_bm25(BM25_PATH)
assert chroma_coll.count() == len(chunks_df)

%time embedder = load_bge(device=best_device())

print(f"\nchunks_df rows : {len(chunks_df):,}")
print(f"ChromaDB count : {chroma_coll.count():,}")
print(f"BM25 chunk_ids : {len(bm25_payload['chunk_ids']):,}")
print(f"BGE device     : {embedder.device}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


CPU times: user 234 ms, sys: 602 ms, total: 837 ms
Wall time: 8.29 s

chunks_df rows : 67,599
ChromaDB count : 67,599
BM25 chunk_ids : 67,599
BGE device     : mps:0


## 5. Stage B — Hybrid retrieval (write `golden_candidates.jsonl`)

For each question, run Hybrid RRF (Chroma + BM25) → top-10 candidates. Write a JSONL row per question with the candidate metadata. **Construction-time bias:** prepend the correct answer to the query so retrieval favours answer-supporting chunks.

In [6]:
def make_candidate_record(row: pd.Series) -> dict:
    options = json.loads(row["options_json"])
    correct_letter = row["answer_idx"]
    correct_text = options[correct_letter]
    construction_query = f"{row['question']}  Answer: {correct_text}"
    cands = hybrid_top_k_with_text(
        construction_query,
        embedder_model=embedder,
        chroma_coll=chroma_coll,
        bm25_payload=bm25_payload,
        chunks_df=chunks_df,
        k=RETRIEVAL_K,
    )
    # Add rank field for the prompt formatter
    for i, c in enumerate(cands, start=1):
        c["rank"] = i
    return {
        "question_id":    int(row.name),
        "meta_info":      row["meta_info"],
        "length_bucket":  row["length_bucket"],
        "question":       row["question"],
        "options":        options,
        "answer_idx":     correct_letter,
        "correct_answer": correct_text,
        "candidates":     cands,
    }

# Stage B is fast (no API calls) — overwrite each run
with open(CANDIDATES_PATH, "w", encoding="utf-8") as fout:
    for _, row in tqdm(list(dev_sample.iterrows()), desc="Stage B retrieval"):
        rec = make_candidate_record(row)
        fout.write(json.dumps(rec, default=str) + "\n")

n_lines = sum(1 for _ in open(CANDIDATES_PATH))
print(f"Saved → {CANDIDATES_PATH}  ({n_lines} rows, {CANDIDATES_PATH.stat().st_size/1024:.1f} KB)")


Stage B retrieval: 100%|██████████| 300/300 [17:36<00:00,  3.52s/it]

Saved → /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/golden/golden_candidates.jsonl  (300 rows, 5391.1 KB)


## 6. Tiny smoke — 1 question through all 3 passes

Run the full pipeline on the first row before kicking off Stage C. Verifies prompts render, GPT-4o returns valid JSON, and validators accept the structure. ~5 s first time, instant cached.

In [7]:
def gpt_json(prompt: str, temperature: float = 0.0, max_tokens: int = 4096):
    """Call openai_complete_full and return (parsed_dict, full_payload, was_cached).
    Raises ValueError if JSON cannot be parsed."""
    payload, was_cached = openai_complete_full(
        prompt, model=CONSTRUCTOR_MODEL,
        temperature=temperature, max_tokens=max_tokens,
        cache_dir=CACHE_DIR,
    )
    parsed = parse_json_with_reasoning_leak(payload["text"])
    if parsed is None:
        raise ValueError(f"JSON malformed: {payload['text'][:200]}")
    return parsed, payload, was_cached


# Smoke: first row of candidates
with open(CANDIDATES_PATH) as f:
    smoke_rec = json.loads(next(f))

# Pass 1
p1_prompt = pass1_prompt(
    smoke_rec["question"], smoke_rec["options"],
    smoke_rec["answer_idx"], smoke_rec["correct_answer"], smoke_rec["candidates"],
)
p1_parsed, p1_payload, p1_cached = gpt_json(p1_prompt, temperature=0.0)
ok, reason = validate_pass1(p1_parsed)
assert ok, f"Pass 1 schema invalid: {reason}"
print(f"Pass 1: cached={p1_cached}  selected={len(p1_parsed['selected_chunks'])}  sufficient={p1_parsed['is_evidence_sufficient']}")

# Pass 2 — only if Pass 1 produced gold context
gold_ctx = p1_parsed["best_gold_context"]
p2_prompt = pass2_prompt(
    smoke_rec["question"], smoke_rec["options"],
    smoke_rec["answer_idx"], smoke_rec["correct_answer"], gold_ctx,
)
p2_parsed, p2_payload, p2_cached = gpt_json(p2_prompt, temperature=0.2)
ok, reason = validate_pass2(p2_parsed)
assert ok, f"Pass 2 schema invalid: {reason}"
print(f"Pass 2: cached={p2_cached}  question_type={p2_parsed['question_type']}  multihop={p2_parsed['requires_multihop']}")

# Pass 3
p3_prompt = pass3_prompt(
    smoke_rec["question"], smoke_rec["answer_idx"], smoke_rec["correct_answer"],
    gold_ctx, p2_parsed["reference_answer"], p2_parsed["reference_explanation"],
)
p3_parsed, p3_payload, p3_cached = gpt_json(p3_prompt, temperature=0.0)
ok, reason = validate_pass3(p3_parsed)
assert ok, f"Pass 3 schema invalid: {reason}"
print(f"Pass 3: cached={p3_cached}  match={p3_parsed['answer_match']}  status={p3_parsed['final_status']}")
print(f"        scores: rel={p3_parsed['evidence_relevance_score']} faith={p3_parsed['faithfulness_score']} qual={p3_parsed['explanation_quality_score']}")

print("\n✓ smoke passed — all 3 passes produced schema-valid JSON")


Pass 1: cached=True  selected=1  sufficient=True
Pass 2: cached=True  question_type=mechanism  multihop=no
Pass 3: cached=True  match=True  status=accepted
        scores: rel=4 faith=4 qual=4

✓ smoke passed — all 3 passes produced schema-valid JSON


## 7. Stage C — Pass 1 evidence selection (write `golden_evidence_selected.jsonl`)

In [8]:
with open(CANDIDATES_PATH) as fin, open(EVIDENCE_PATH, "w", encoding="utf-8") as fout:
    for line in tqdm(list(fin), desc="Pass 1 evidence selection"):
        rec = json.loads(line)
        prompt = pass1_prompt(
            rec["question"], rec["options"],
            rec["answer_idx"], rec["correct_answer"], rec["candidates"],
        )
        try:
            parsed, payload, was_cached = gpt_json(prompt, temperature=PASS1_TEMP, max_tokens=PASS1_MAX_TOKENS)
            ok, reason = validate_pass1(parsed)
            if not ok:
                rec["evidence_selection"] = parsed
                rec["evidence_selection_status"] = f"schema_fail: {reason}"
            else:
                rec["evidence_selection"] = parsed
                rec["evidence_selection_status"] = "ok"
            rec["evidence_selection_usage"] = payload["usage"]
            rec["evidence_selection_cached"] = was_cached
        except Exception as e:
            rec["evidence_selection"] = None
            rec["evidence_selection_status"] = f"error: {type(e).__name__}: {e}"
        fout.write(json.dumps(rec, default=str) + "\n")

# Summary
status_counts = Counter()
for line in open(EVIDENCE_PATH):
    status_counts[json.loads(line)["evidence_selection_status"]] += 1
print(f"Saved → {EVIDENCE_PATH}  ({EVIDENCE_PATH.stat().st_size/1024:.1f} KB)")
for s, c in status_counts.most_common():
    print(f"  {c:>3}  {s}")


Pass 1 evidence selection: 100%|██████████| 300/300 [25:29<00:00,  5.10s/it]

Saved → /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/golden/golden_evidence_selected.jsonl  (6038.9 KB)
  294  ok
    3  schema_fail: best_gold_context must be non-empty string
    1  schema_fail: selected_chunks must be list of 1-5
    1  schema_fail: evidence_keywords must be list of 3-10
    1  error: ValueError: JSON malformed: ```json
{
  "selected_chunks": [
    {
      "chunk_id": "Immunology_Janeway_chunk_00314",
      "book_name": "Immunology_Janeway",
      "support_level": "strong",
      "reason": "This passage expla


## 8. Stage D — Pass 2 reference answer (write `golden_with_references.jsonl`)

Skips rows where Pass 1 errored or returned no `best_gold_context`.

In [9]:
with open(EVIDENCE_PATH) as fin, open(REFERENCES_PATH, "w", encoding="utf-8") as fout:
    for line in tqdm(list(fin), desc="Pass 2 reference answers"):
        rec = json.loads(line)
        ev = rec.get("evidence_selection") or {}
        gold_context = ev.get("best_gold_context", "") if rec.get("evidence_selection_status") == "ok" else ""

        if not gold_context:
            rec["reference"] = None
            rec["reference_status"] = "skipped: no gold context"
            fout.write(json.dumps(rec, default=str) + "\n")
            continue

        prompt = pass2_prompt(
            rec["question"], rec["options"],
            rec["answer_idx"], rec["correct_answer"], gold_context,
        )
        try:
            parsed, payload, was_cached = gpt_json(prompt, temperature=PASS2_TEMP, max_tokens=PASS2_MAX_TOKENS)
            ok, reason = validate_pass2(parsed)
            if not ok:
                rec["reference"] = parsed
                rec["reference_status"] = f"schema_fail: {reason}"
            else:
                rec["reference"] = parsed
                rec["reference_status"] = "ok"
            rec["reference_usage"] = payload["usage"]
            rec["reference_cached"] = was_cached
        except Exception as e:
            rec["reference"] = None
            rec["reference_status"] = f"error: {type(e).__name__}: {e}"
        fout.write(json.dumps(rec, default=str) + "\n")

status_counts = Counter()
for line in open(REFERENCES_PATH):
    status_counts[json.loads(line)["reference_status"]] += 1
print(f"Saved → {REFERENCES_PATH}  ({REFERENCES_PATH.stat().st_size/1024:.1f} KB)")
for s, c in status_counts.most_common():
    print(f"  {c:>3}  {s}")


Pass 2 reference answers: 100%|██████████| 300/300 [15:46<00:00,  3.15s/it]

Saved → /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/golden/golden_with_references.jsonl  (6406.1 KB)
  294  ok
    6  skipped: no gold context


## 9. Stage E — Pass 3 validation (write `golden_validated.jsonl`)

Skips rows where Pass 2 didn't produce a reference.

In [10]:
with open(REFERENCES_PATH) as fin, open(VALIDATED_PATH, "w", encoding="utf-8") as fout:
    for line in tqdm(list(fin), desc="Pass 3 validation"):
        rec = json.loads(line)
        ref = rec.get("reference") or {}
        ev = rec.get("evidence_selection") or {}

        if rec.get("reference_status") != "ok":
            rec["validation"] = None
            rec["validation_status"] = "skipped"
            fout.write(json.dumps(rec, default=str) + "\n")
            continue

        prompt = pass3_prompt(
            rec["question"], rec["answer_idx"], rec["correct_answer"],
            ev.get("best_gold_context", ""),
            ref.get("reference_answer", ""), ref.get("reference_explanation", ""),
        )
        try:
            parsed, payload, was_cached = gpt_json(prompt, temperature=PASS3_TEMP, max_tokens=PASS3_MAX_TOKENS)
            ok, reason = validate_pass3(parsed)
            if not ok:
                rec["validation"] = parsed
                rec["validation_status"] = f"schema_fail: {reason}"
            else:
                rec["validation"] = parsed
                rec["validation_status"] = "ok"
            rec["validation_usage"] = payload["usage"]
            rec["validation_cached"] = was_cached
        except Exception as e:
            rec["validation"] = None
            rec["validation_status"] = f"error: {type(e).__name__}: {e}"
        fout.write(json.dumps(rec, default=str) + "\n")

# Status distribution
statuses = Counter()
for line in open(VALIDATED_PATH):
    rec = json.loads(line)
    v = rec.get("validation") or {}
    statuses[v.get("final_status", rec.get("validation_status", "missing"))] += 1
print(f"Saved → {VALIDATED_PATH}  ({VALIDATED_PATH.stat().st_size/1024:.1f} KB)")
for s, c in statuses.most_common():
    print(f"  {c:>3}  {s}")


Pass 3 validation:   0%|          | 0/300 [00:00<?, ?it/s]

Pass 3 validation: 100%|██████████| 300/300 [09:36<00:00,  1.92s/it]

Saved → /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/golden/golden_validated.jsonl  (6578.3 KB)
  238  accepted
   49  needs_review
    7  rejected
    6  skipped


## 10. Stage F — Audit + final accept/needs_review/dropped split

Pure-Python audit (no LLM):
1. Every cited `chunk_id` must resolve in `chunks.parquet`
2. The gold answer text must appear (case-insensitive) in `reference_answer`
3. At least one `evidence_keyword` must appear in the gold context
4. Pass-3 `answer_match` must be `True`

Failures downgrade `accepted` → `needs_review`. Pass-3 rejections / Pass-1/2 errors / skipped → `dropped`.

In [11]:
def audit_one(rec: dict) -> tuple[str, list[str], dict | None]:
    """Return (post_audit_status, audit_flags, golden_record_or_None)."""
    if rec.get("validation_status") != "ok":
        return "rejected", [rec.get("validation_status", "missing")], None

    v = rec["validation"]
    ev = rec["evidence_selection"]
    ref = rec["reference"]
    flags: list[str] = []

    selected_chunks = ev.get("selected_chunks", [])
    cited_ids = [c.get("chunk_id", "") for c in selected_chunks]
    unresolved = [cid for cid in cited_ids if cid and cid not in chunk_text_by_id]
    if unresolved:
        flags.append(f"unresolved_chunk_ids:{len(unresolved)}")

    if rec["correct_answer"].lower() not in ref.get("reference_answer", "").lower():
        flags.append("gold_answer_not_in_reference")

    gold_ctx_lower = (ev.get("best_gold_context") or "").lower()
    keyword_hits = sum(1 for kw in ev.get("evidence_keywords", []) if kw.lower() in gold_ctx_lower)
    if keyword_hits == 0:
        flags.append("no_keyword_in_context")

    if not v.get("answer_match"):
        flags.append("pass3_answer_match_false")

    base_status = v["final_status"]
    if flags and base_status == "accepted":
        status = "needs_review"
    else:
        status = base_status

    if status in ("accepted", "needs_review"):
        golden_row = {
            "question_id":                          rec["question_id"],
            "meta_info":                            rec["meta_info"],
            "question":                             rec["question"],
            "options":                              rec["options"],
            "gold_answer_letter":                   rec["answer_idx"],
            "gold_answer_text":                     rec["correct_answer"],
            "gold_chunks":                          [c.get("chunk_id") for c in selected_chunks],
            "gold_context":                         ev.get("best_gold_context"),
            "gold_chunks_metadata":                 selected_chunks,
            "evidence_keywords":                    ev.get("evidence_keywords"),
            "reference_answer":                     ref["reference_answer"],
            "reference_explanation":                ref["reference_explanation"],
            "why_other_options_are_less_suitable": ref["why_other_options_are_less_suitable"],
            "hallucination_check_points":           ref["hallucination_check_points"],
            "question_type":                        ref["question_type"].lower(),
            "requires_multihop":                    ref["requires_multihop"].lower(),
            "answer_match":                         v["answer_match"],
            "evidence_relevance_score":             v["evidence_relevance_score"],
            "faithfulness_score":                   v["faithfulness_score"],
            "explanation_quality_score":            v["explanation_quality_score"],
            "hallucination_risk":                   v["hallucination_risk"],
            "final_status":                         status,
            "audit_flags":                          flags,
        }
        return status, flags, golden_row
    return status, flags, None


accepted, needs_review, dropped = [], [], []
for line in open(VALIDATED_PATH):
    rec = json.loads(line)
    status, flags, golden_row = audit_one(rec)
    if status == "accepted":
        accepted.append(golden_row)
    elif status == "needs_review":
        needs_review.append(golden_row)
    else:
        dropped.append({
            "question_id":      rec["question_id"],
            "question":         rec["question"],
            "answer_idx":       rec["answer_idx"],
            "drop_reason":      flags or [rec.get("validation_status", "missing")],
            "evidence_status":  rec.get("evidence_selection_status"),
            "reference_status": rec.get("reference_status"),
            "validation_status": rec.get("validation_status"),
        })


def write_jsonl(path: Path, rows: list[dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, default=str, ensure_ascii=False) + "\n")

write_jsonl(ACCEPTED_PATH, accepted)
write_jsonl(NEEDS_REVIEW_PATH, needs_review)
write_jsonl(DROPPED_PATH, dropped)

print("Stage F written:")
for path, rows in [(ACCEPTED_PATH, accepted), (NEEDS_REVIEW_PATH, needs_review), (DROPPED_PATH, dropped)]:
    print(f"  {path.name:<38}  {len(rows):>3} rows  ({path.stat().st_size/1024:.1f} KB)")


Stage F written:
  golden_main_accepted.jsonl              234 rows  (1015.1 KB)
  golden_main_needs_review.jsonl           53 rows  (223.7 KB)
  golden_main_dropped.jsonl                13 rows  (15.1 KB)


## 11. Quality gates + real-cost summary

In [12]:
# Reload to compute gates from the staged files (clean source of truth)
n = N
n_accepted    = len(accepted)
n_needs_rev   = len(needs_review)
n_dropped     = len(dropped)
n_salvageable = n_accepted + n_needs_rev

# JSON malformation = any pass returned None or schema_fail
malformed = 0
total_passes = 0
n_pass1_ok = n_pass2_ok = n_pass3_ok = 0
n_sufficient = 0
n_with_p1 = 0
multihop_yes = 0

for line in open(VALIDATED_PATH):
    rec = json.loads(line)
    for k in ("evidence_selection", "reference", "validation"):
        status_key = {"evidence_selection": "evidence_selection_status",
                      "reference": "reference_status",
                      "validation": "validation_status"}[k]
        st = rec.get(status_key, "")
        total_passes += 1
        if rec.get(k) is None and st.startswith(("error", "schema_fail")):
            malformed += 1
    if rec.get("evidence_selection_status") == "ok":
        n_pass1_ok += 1
        n_with_p1 += 1
        if (rec.get("evidence_selection") or {}).get("is_evidence_sufficient"):
            n_sufficient += 1
    if rec.get("reference_status") == "ok":
        n_pass2_ok += 1
    if rec.get("validation_status") == "ok":
        n_pass3_ok += 1
    ref = rec.get("reference") or {}
    if ref.get("requires_multihop", "").lower() == "yes":
        multihop_yes += 1

malformation_rate = malformed / total_passes if total_passes else 0.0
sufficiency_rate  = n_sufficient / n_with_p1 if n_with_p1 else 0.0
multihop_rate     = multihop_yes / n
all_resolve = all(
    cid in chunk_text_by_id
    for r in (accepted + needs_review)
    for cid in r["gold_chunks"] if cid
)

print(f"=== QUALITY GATES — {CONSTRUCTOR_MODEL}  (stage={STAGE}, N={n}) ===")
gates = [
    ("Accept rate           >= 80%", f"{n_accepted}/{n} = {n_accepted/n:.1%}",            n_accepted/n >= 0.80),
    ("JSON malformation     <  5%", f"{malformed}/{total_passes} = {malformation_rate:.2%}", malformation_rate < 0.05),
    ("Pass-1 sufficiency    >= 90%", f"{n_sufficient}/{n_with_p1} = {sufficiency_rate:.1%}", sufficiency_rate >= 0.90),
    ("All chunk_ids resolve",         "yes" if all_resolve else "no",                      all_resolve),
    ("requires_multihop yes <  60%", f"{multihop_yes}/{n} = {multihop_rate:.1%}",          multihop_rate < 0.60),
]
for name, value, ok in gates:
    print(f"  {'✓' if ok else '✗'}  {name:<35}  {value}")
print()
print(f"Status mix: accepted={n_accepted}  needs_review={n_needs_rev}  dropped={n_dropped}  "
      f"salvageable={n_salvageable}/{n} ({n_salvageable/n:.1%})")
print()

# === Real cost (sum usage from all 3 passes across all rows) ===
total_in  = 0
total_out = 0
total_cached = 0
total_calls = 0
for line in open(VALIDATED_PATH):
    rec = json.loads(line)
    for usage_key, cached_key in [
        ("evidence_selection_usage", "evidence_selection_cached"),
        ("reference_usage", "reference_cached"),
        ("validation_usage", "validation_cached"),
    ]:
        u = rec.get(usage_key)
        if u:
            total_in  += int(u.get("prompt_tokens") or 0)
            total_out += int(u.get("completion_tokens") or 0)
            total_calls += 1
            if rec.get(cached_key): total_cached += 1

# OpenAI GPT-4o pricing
cost_paid_openai = total_in * (2.50 / 1_000_000) + total_out * (10.00 / 1_000_000)
extrap_300 = cost_paid_openai * (PRODUCTION_N / n) if STAGE == "pilot" else cost_paid_openai

print("=== REAL COST ===")
print(f"  total prompt tokens   : {total_in:,}")
print(f"  total completion tokens: {total_out:,}")
print(f"  total tokens          : {total_in + total_out:,}")
print(f"  total LLM calls       : {total_calls}")
print(f"  cache hits            : {total_cached} / {total_calls}")
print()
print(f"  Cost on OpenAI ({STAGE} run): ${cost_paid_openai:.4f}")
if STAGE == "pilot":
    print(f"  Extrapolated to 300 rows : ${extrap_300:.2f}")


=== QUALITY GATES — gpt-4o  (stage=production, N=300) ===
  ✗  Accept rate           >= 80%         234/300 = 78.0%
  ✓  JSON malformation     <  5%          1/900 = 0.11%
  ✓  Pass-1 sufficiency    >= 90%         280/294 = 95.2%
  ✓  All chunk_ids resolve                yes
  ✓  requires_multihop yes <  60%         18/300 = 6.0%

Status mix: accepted=234  needs_review=53  dropped=13  salvageable=287/300 (95.7%)

=== REAL COST ===
  total prompt tokens   : 1,646,183
  total completion tokens: 248,960
  total tokens          : 1,895,143
  total LLM calls       : 887
  cache hits            : 138 / 887

  Cost on OpenAI (production run): $6.6051


## 12. Sample outputs — eyeball check

Print 2 accepted + 1 needs_review + 1 dropped (if available) to spot-check quality before scaling.

In [13]:
def show(g, status):
    print(f"--- {status}  audit_flags={g.get('audit_flags', [])}  "
          f"P3 scores: rel={g['evidence_relevance_score']} faith={g['faithfulness_score']} qual={g['explanation_quality_score']} ---")
    print(f"Q ({g['meta_info']}): {g['question'][:180]}...")
    print(f"A ({g['gold_answer_letter']}): {g['gold_answer_text']}")
    print(f"reference_answer:    {g['reference_answer']}")
    print(f"reference_explan:    {g['reference_explanation'][:280]}...")
    print(f"check points:        {g['hallucination_check_points']}")
    print(f"requires_multihop:   {g['requires_multihop']}    question_type: {g['question_type']}")
    print()

for r in accepted[:2]:
    show(r, "accepted")
if needs_review:
    print("=== needs_review sample ===")
    show(needs_review[0], "needs_review")
if dropped:
    print("=== dropped sample ===")
    d = dropped[0]
    print(f"question_id={d['question_id']}  drop_reason={d['drop_reason']}")
    print(f"  evidence_status:  {d.get('evidence_status')}")
    print(f"  reference_status: {d.get('reference_status')}")
    print(f"  validation_status: {d.get('validation_status')}")
    print(f"  Q: {d['question'][:180]}...")


--- accepted  audit_flags=[]  P3 scores: rel=4 faith=4 qual=4 ---
Q (step1): A 12-year-old boy who recently immigrated from Namibia is being evaluated for exertional shortness of breath and joint pain for the past month. His mother reports that he used to p...
A (B): Type II hypersensitivity reaction
reference_answer:    Type II hypersensitivity reaction
reference_explan:    The boy's symptoms and laboratory findings suggest rheumatic fever, which is associated with a preceding group A streptococcal infection, as indicated by the elevated antistreptolysin O titer. Rheumatic fever is a result of a Type II hypersensitivity reaction, where antibodies ta...
check points:        ['The boy has elevated antistreptolysin O titer.', 'Rheumatic fever is a result of a Type II hypersensitivity reaction.', 'The presence of polyarthritis and subcutaneous nodules supports rheumatic fever.']
requires_multihop:   no    question_type: mechanism

--- accepted  audit_flags=[]  P3 scores: rel=4 faith=5 qua

---

**Done.** Next steps based on §11 output:

**If gates pass** (accept rate ≥ 80 %, malformation < 5 %, multi-hop < 60 %, etc.):
1. Send the §11 output to me.
2. Change `STAGE = "pilot"` → `STAGE = "production"` in §1.
3. **Restart Kernel & Run All.** First 50 questions are cached → instant. New 250 cost ~$5.50.
4. After production run, send §11 output again — I'll write `docs/output_notes/` and propagate the constructor lock across plan / tech_stack / architecture / memory.

**If gates fail:**
- Multi-hop > 60 %: tighten the prompt further in `src/generation/golden_prompts.py`. Re-run only Pass 2 (Pass 1 stays cached).
- Sufficiency < 90 %: loosen the Pass 1 sufficiency criterion. Re-run Pass 1 (downstream re-fires too because Pass 1's output changes).
- Accept rate < 80 % from audit flags: inspect the audit-flagged needs_review rows; the issues are usually `gold_answer_not_in_reference` (Pass 2 phrasing) or `no_keyword_in_context` (Pass 1 keyword extraction). Tune the relevant pass.

The staged-JSONL pattern means each fix is cheap — only the affected pass re-fires; the rest is read from disk.